# Next-Word Predictor using LSTM

This notebook trains an LSTM-based next-word prediction model and includes:
- Train / validation split so you can monitor **validation accuracy** during training
- Early stopping to avoid overfitting
- Training curves (loss & accuracy) for train vs validation
- A `predict_next_word` and `generate_text` function so you can **test** the model interactively


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)

## 2. Training Text

Replace / extend `faqs` with your own corpus. The more text you provide, the better the model will generalize — a single short article (like below) is fine for learning the pipeline, but will overfit quickly since the vocabulary and number of training sequences are small.

In [ ]:
faqs="""# Artificial Intelligence and the Art of Predicting Words: Building a Next-Word Predictor with LSTM

## Introduction

Artificial Intelligence (AI) has quietly woven itself into the fabric of everyday life. From the moment we type a search query and see suggestions appear before we finish our sentence, to the autocomplete feature on our smartphones that seems to read our minds, AI-driven language prediction is one of the most visible and widely used applications of machine learning. At the heart of many such systems lies a class of neural networks called Recurrent Neural Networks (RNNs), and more specifically, a powerful variant known as Long Short-Term Memory networks, or LSTMs. This article explores the broader landscape of AI, the specific challenge of next-word prediction, and how LSTM networks are uniquely suited to solve it — a task you are now undertaking yourself.

## The Broader Context: AI and Natural Language Processing

Artificial Intelligence, in its simplest definition, refers to the simulation of human intelligence processes by machines, particularly computer systems. These processes include learning, reasoning, problem-solving, perception, and language understanding. Within this vast field, Natural Language Processing (NLP) is the subdomain concerned with enabling computers to understand, interpret, and generate human language.

Language is inherently sequential and contextual. The meaning of a word often depends heavily on the words that precede it. Consider the sentence "I went to the bank to withdraw some ___." Most humans would instinctively predict "money" or "cash" as the next word, because we understand the context established by "bank" and "withdraw." This kind of contextual reasoning, which feels effortless to humans, is remarkably difficult for machines. Teaching a computer to grasp these dependencies — to look back at a sequence of words and use that history to make an informed guess about what comes next — is precisely the problem that next-word prediction models are designed to solve.

Next-word prediction sits at the foundation of many larger NLP applications: autocomplete systems, chatbots, machine translation, speech recognition, and even the large language models that power modern conversational AI. Understanding how to build a next-word predictor is, in many ways, understanding the building block from which much larger and more sophisticated language systems are constructed.

## Why Traditional Approaches Fall Short

Before neural networks became the dominant approach, language modeling relied heavily on statistical methods, particularly n-gram models. An n-gram model predicts the next word based on the previous n-1 words by calculating probabilities from a large corpus of text. For example, a trigram model would look at the previous two words and estimate the probability of each possible next word based on how frequently that three-word sequence appeared in the training data.

While n-gram models are simple and computationally efficient, they suffer from significant limitations. First, they cannot capture long-range dependencies — a trigram model has no memory of anything beyond the last two words, so it cannot use information from much earlier in a sentence or paragraph. Second, they suffer from the curse of dimensionality: as n increases to capture more context, the number of possible word combinations grows exponentially, making the model sparse and unreliable for combinations that were rarely or never seen during training. Third, n-gram models treat words as discrete, unrelated symbols, failing to capture semantic similarity between words like "happy" and "joyful."

These limitations paved the way for neural network-based approaches, which can learn continuous vector representations of words (embeddings) and, crucially, can be designed to maintain a memory of prior context over much longer sequences.

## Enter Recurrent Neural Networks

A Recurrent Neural Network is a type of neural network specifically designed to handle sequential data. Unlike traditional feedforward neural networks, which process inputs independently, RNNs maintain a hidden state that gets updated at each time step, carrying information forward from one step to the next. This makes them naturally suited for tasks involving sequences, such as time series analysis, speech recognition, and, of course, language modeling.

In the context of next-word prediction, an RNN processes a sentence one word at a time. At each step, it takes the current word (represented as a vector) and the hidden state from the previous step, and produces a new hidden state. This hidden state theoretically encodes information about all the words seen so far. After processing the entire input sequence, the network's final hidden state can be used to predict the probability distribution over the vocabulary for the next word.

However, vanilla RNNs have a serious flaw: the vanishing gradient problem. During training, neural networks learn by backpropagating error gradients through the network to update weights. In RNNs, this backpropagation happens through time — across every time step in the sequence. When sequences are long, gradients can shrink exponentially as they are propagated backward, becoming so small that earlier layers effectively stop learning. This means vanilla RNNs struggle to learn dependencies that span more than a few words, which is a serious limitation for language, where context from many words earlier can be crucial to correctly predicting what comes next.

## LSTM: A Solution to Long-Range Memory

Long Short-Term Memory networks, introduced by Sepp Hochreiter and Jürgen Schmidhuber in 1997, were specifically designed to address the vanishing gradient problem and enable networks to learn long-range dependencies. LSTMs achieve this through a more sophisticated internal architecture that includes a "cell state" and a series of gates that regulate the flow of information.

The core innovation of the LSTM is the cell state, which acts like a conveyor belt running through the entire sequence. Information can be added to or removed from the cell state through three carefully regulated gates:

The Forget Gate decides what information from the previous cell state should be discarded. It looks at the previous hidden state and the current input, and outputs a value between 0 and 1 for each number in the cell state — where 1 means "completely keep this" and 0 means "completely forget this."

The Input Gate decides what new information should be stored in the cell state. This involves two parts: a sigmoid layer that decides which values to update, and a tanh layer that creates a vector of new candidate values that could be added to the state.

The Output Gate determines what the next hidden state should be, based on the updated cell state. This hidden state is what gets passed to the next time step and is also used to make predictions at the current step.

Because of this gating mechanism, LSTMs can selectively remember important information over long sequences while discarding irrelevant details, and gradients can flow through the cell state with much less degradation than in vanilla RNNs. This makes LSTMs far more effective for tasks like next-word prediction, where understanding context from many words earlier in a sentence — or even a paragraph — can meaningfully change what word should come next.

## Building a Next-Word Predictor: The Practical Pipeline

Since you're building your own LSTM-based next-word predictor, it's worth walking through the typical pipeline you'll likely follow, along with some practical considerations at each stage.

1. Data Collection and Preprocessing
The first step is gathering a text corpus — this could be anything from a collection of books, articles, movie scripts, or domain-specific text depending on your use case. The raw text needs to be cleaned: lowercasing (usually), removing special characters, handling punctuation consistently, and tokenizing the text into individual words. Tokenization converts a continuous string of text into a list of discrete units (tokens) that the model can work with.

2. Building a Vocabulary
Once tokenized, you build a vocabulary — a mapping from each unique word to a numerical index. This is necessary because neural networks operate on numbers, not raw text. You'll typically also decide on a vocabulary size cutoff, since real-world corpora can have extremely large vocabularies, and rare words are often replaced with a special "unknown" token to keep the model manageable.

3. Creating Training Sequences
Next-word prediction is framed as a supervised learning problem: given a sequence of words, predict the next one. This means you'll slide a window across your tokenized text, creating input sequences paired with the target label (the word that follows). Padding is often applied to ensure sequences have consistent lengths for batch processing.

4. Word Embeddings
Rather than feeding raw word indices into the LSTM, it's standard practice to first pass words through an embedding layer. This layer learns a dense vector representation for each word, capturing semantic relationships — words used in similar contexts end up with similar vector representations.

5. The LSTM Architecture
A typical architecture for next-word prediction consists of an embedding layer, followed by one or more LSTM layers, followed by a dense output layer with a softmax activation function. The softmax layer outputs a probability distribution over the entire vocabulary, and the word with the highest probability becomes the predicted next word.

6. Training the Model
Training involves feeding batches of input sequences through the network, comparing the predicted probability distribution against the actual next word using a loss function, typically categorical cross-entropy, and updating the model's weights using an optimizer such as Adam. It's important to monitor both training loss and validation loss to catch overfitting.

7. Evaluation and Generation
Once trained, the model can be evaluated using metrics like perplexity. For practical use, you can generate text by feeding a seed phrase into the model, predicting the next word, appending it to the sequence, and repeating the process.

## Practical Challenges to Anticipate

Building an effective next-word predictor comes with real challenges. Overfitting is common, especially with smaller datasets, since LSTMs have many parameters and can memorize training sequences rather than learning generalizable patterns. Techniques like dropout, regularization, and early stopping can help mitigate this. Computational cost is another consideration. Vocabulary size also matters significantly. Finally, LSTMs still have practical limits on how far back they can effectively remember, which is part of why newer architectures like Transformers have become dominant for large-scale language modeling.

## Conclusion

Building a next-word predictor using LSTM networks is both a practical exercise and a window into the fundamental challenges of natural language processing. LSTMs, with their elegant gating mechanisms, solved a critical problem in sequence modeling and paved the way for much of the progress in NLP that followed.
"""

## 3. Tokenization

Fit a Keras `Tokenizer` on the corpus to build a word-to-index vocabulary.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([faqs])

vocab_size = len(tokenizer.word_index) + 1  # +1 because Keras reserves index 0 for padding
print("Vocabulary size (including padding index 0):", vocab_size)

## 4. Build N-gram Input Sequences

For every line, we create progressively longer subsequences — e.g. for the tokens `[a, b, c, d]` we generate `[a, b]`, `[a, b, c]`, `[a, b, c, d]`. The last token in each subsequence is the label (the word to predict), and everything before it is the input context.

In [ ]:
input_sequences = []
for sentence in faqs.split('\n'):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i + 1])

print("Number of training sequences:", len(input_sequences))

## 5. Pad Sequences

In [ ]:
max_len = max([len(x) for x in input_sequences])
print("Max sequence length:", max_len)

padded_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')
padded_input_sequences.shape

## 6. Split Into Features (X) and Labels (y)

In [ ]:
X = padded_input_sequences[:, :-1]
y = padded_input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
y = to_categorical(y, num_classes=vocab_size)
print("y shape after one-hot encoding:", y.shape)

## 7. Train / Validation Split

This is the key addition that lets you monitor **validation accuracy** during training, instead of only seeing training accuracy. We hold out 15% of the sequences as a validation set.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42
)

print("X_train:", X_train.shape, " X_val:", X_val.shape)
print("y_train:", y_train.shape, " y_val:", y_val.shape)

## 8. Build the Model

Note the embedding input dimension, embedding input length, and final dense layer size are now all derived automatically from your data (`vocab_size`, `max_len`) instead of being hardcoded — so this notebook keeps working correctly if you change the training text.

In [ ]:
model = Sequential([
    Embedding(vocab_size, 100, input_length=max_len - 1),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(150),
    Dropout(0.2),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## 9. Callbacks: Early Stopping + Best-Model Checkpoint

- `EarlyStopping` watches validation loss and stops training once it stops improving, restoring the best weights.
- `ModelCheckpoint` saves the best-performing model (by validation accuracy) to disk as `best_model.keras`.

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

## 10. Train the Model

`validation_data=(X_val, y_val)` is what makes Keras report `val_loss` and `val_accuracy` alongside the training metrics after every epoch.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

## 11. Plot Training vs Validation Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

## 12. Evaluate on the Validation Set

A final, single-number summary of how the model performs on data it did not train on.

In [ ]:
val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

## 13. Testing: Predict the Next Word

Given some input text, this function tokenizes it, pads it to the same `max_len` the model was trained on, and returns the model's top predicted next word (plus its top-k alternatives).

In [ ]:
def predict_next_word(model, tokenizer, text, max_len, top_k=3):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len - 1, padding='pre')

    predicted_probs = model.predict(token_list, verbose=0)[0]
    top_indices = predicted_probs.argsort()[-top_k:][::-1]

    index_to_word = {index: word for word, index in tokenizer.word_index.items()}

    results = []
    for idx in top_indices:
        word = index_to_word.get(idx, '')
        if word:
            results.append((word, float(predicted_probs[idx])))
    return results


# Example usage
sample_text = "artificial intelligence has"
predictions = predict_next_word(model, tokenizer, sample_text, max_len)
print(f"Input: '{sample_text}'")
print("Top predictions (word, probability):")
for word, prob in predictions:
    print(f"  {word}: {prob:.4f}")

## 14. Testing: Generate a Longer Passage

Repeatedly predicts the next word and appends it to the input, letting you see how the model generates multi-word continuations from a seed phrase.

In [ ]:
def generate_text(model, tokenizer, seed_text, max_len, next_words=15):
    text = seed_text
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1, padding='pre')

        predicted_index = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ''
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        if not output_word:
            break

        text += ' ' + output_word

    return text


# Example usage
generated = generate_text(model, tokenizer, "long short term memory", max_len, next_words=15)
print(generated)

## 15. Try Your Own Inputs

Edit `my_seed_text` below and re-run this cell to test the model interactively.

In [ ]:
my_seed_text = "recurrent neural"
print(generate_text(model, tokenizer, my_seed_text, max_len, next_words=10))

## 16. (Optional) Load the Best Saved Model Later

Since `ModelCheckpoint` saved the best model to `best_model.keras`, you can reload it in a future session without retraining.

In [ ]:
from tensorflow.keras.models import load_model

# best_model = load_model('best_model.keras')
# print(generate_text(best_model, tokenizer, "artificial intelligence", max_len, next_words=10))